Pubmed API connection - Abstracts

In [1]:
# Import required libraries for PubMed API connection
import requests
import xml.etree.ElementTree as ET
from urllib.parse import quote
import time
import pandas as pd
from typing import List, Dict, Optional
import json

# Base URLs for PubMed API
ESEARCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
EFETCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

print("PubMed API libraries imported successfully!")

PubMed API libraries imported successfully!


In [2]:
class PubMedSearcher:
    """
    A class to search PubMed and retrieve abstracts for scientific papers.
    """
    
    def __init__(self, email: Optional[str] = None, api_key: Optional[str] = None):
        """
        Initialize PubMed searcher.
        
        Args:
            email: Your email (recommended by NCBI for identification)
            api_key: NCBI API key for higher rate limits (optional)
        """
        self.email = email
        self.api_key = api_key
        self.base_params = {}
        
        if email:
            self.base_params['email'] = email
        if api_key:
            self.base_params['api_key'] = api_key
    
    def search_papers(self, query: str, max_results: int = 20, sort: str = "relevance") -> List[str]:
        """
        Search PubMed for papers matching the query.
        
        Args:
            query: Search query (e.g., "diabetes mellitus causal inference")
            max_results: Maximum number of results to return
            sort: Sort order ("relevance", "pub_date", "Author", etc.)
        
        Returns:
            List of PubMed IDs (PMIDs)
        """
        params = {
            **self.base_params,
            'db': 'pubmed',
            'term': query,
            'retmax': max_results,
            'sort': sort,
            'retmode': 'xml'
        }
        
        try:
            response = requests.get(ESEARCH_URL, params=params)
            response.raise_for_status()
            
            # Parse XML response
            root = ET.fromstring(response.content)
            
            # Extract PMIDs
            pmids = []
            for id_elem in root.findall('.//Id'):
                pmids.append(id_elem.text)
            
            print(f"Found {len(pmids)} papers for query: '{query}'")
            return pmids
            
        except Exception as e:
            print(f"Error searching PubMed: {e}")
            return []
    
    def fetch_abstracts(self, pmids: List[str]) -> List[Dict]:
        """
        Fetch detailed information including abstracts for given PMIDs.
        
        Args:
            pmids: List of PubMed IDs
        
        Returns:
            List of dictionaries containing paper information
        """
        if not pmids:
            return []
        
        # Convert list to comma-separated string
        id_string = ','.join(pmids)
        
        params = {
            **self.base_params,
            'db': 'pubmed',
            'id': id_string,
            'retmode': 'xml',
            'rettype': 'abstract'
        }
        
        try:
            response = requests.get(EFETCH_URL, params=params)
            response.raise_for_status()
            
            # Parse XML response
            root = ET.fromstring(response.content)
            
            papers = []
            for article in root.findall('.//PubmedArticle'):
                paper_info = self._extract_paper_info(article)
                if paper_info:
                    papers.append(paper_info)
            
            print(f"Successfully retrieved {len(papers)} abstracts")
            return papers
            
        except Exception as e:
            print(f"Error fetching abstracts: {e}")
            return []
    
    def _extract_paper_info(self, article_elem) -> Optional[Dict]:
        """Extract relevant information from a PubmedArticle XML element."""
        try:
            paper = {}
            
            # Extract PMID
            pmid_elem = article_elem.find('.//PMID')
            paper['pmid'] = pmid_elem.text if pmid_elem is not None else 'Unknown'
            
            # Extract title
            title_elem = article_elem.find('.//ArticleTitle')
            paper['title'] = title_elem.text if title_elem is not None else 'No title'
            
            # Extract abstract
            abstract_parts = []
            abstract_elem = article_elem.find('.//Abstract')
            if abstract_elem is not None:
                for text_elem in abstract_elem.findall('.//AbstractText'):
                    label = text_elem.get('Label', '')
                    text = text_elem.text or ''
                    if label:
                        abstract_parts.append(f"{label}: {text}")
                    else:
                        abstract_parts.append(text)
            
            paper['abstract'] = ' '.join(abstract_parts) if abstract_parts else 'No abstract available'
            
            # Extract authors
            authors = []
            author_list = article_elem.find('.//AuthorList')
            if author_list is not None:
                for author in author_list.findall('.//Author'):
                    last_name = author.find('.//LastName')
                    first_name = author.find('.//ForeName')
                    if last_name is not None and first_name is not None:
                        authors.append(f"{first_name.text} {last_name.text}")
                    elif last_name is not None:
                        authors.append(last_name.text)
            
            paper['authors'] = '; '.join(authors) if authors else 'Unknown authors'
            
            # Extract journal
            journal_elem = article_elem.find('.//Journal/Title')
            paper['journal'] = journal_elem.text if journal_elem is not None else 'Unknown journal'
            
            # Extract publication date
            pub_date = article_elem.find('.//PubDate')
            if pub_date is not None:
                year = pub_date.find('.//Year')
                month = pub_date.find('.//Month')
                paper['pub_date'] = f"{year.text if year is not None else 'Unknown'}-{month.text if month is not None else 'Unknown'}"
            else:
                paper['pub_date'] = 'Unknown date'
            
            # Extract DOI
            doi_elem = article_elem.find('.//ELocationID[@EIdType="doi"]')
            paper['doi'] = doi_elem.text if doi_elem is not None else 'No DOI'
            
            return paper
            
        except Exception as e:
            print(f"Error extracting paper info: {e}")
            return None
    
    def search_and_fetch(self, query: str, max_results: int = 20) -> List[Dict]:
        """
        Complete workflow: search and fetch abstracts in one call.
        
        Args:
            query: Search query
            max_results: Maximum number of results
        
        Returns:
            List of paper dictionaries with abstracts
        """
        print(f"Searching PubMed for: '{query}'")
        pmids = self.search_papers(query, max_results)
        
        if pmids:
            # Add small delay to be respectful to NCBI servers
            time.sleep(0.5)
            papers = self.fetch_abstracts(pmids)
            return papers
        else:
            return []

print("PubMedSearcher class defined successfully!")

PubMedSearcher class defined successfully!


## Example Usage

Now let's create an instance and test the PubMed API connection with some example queries.

In [ ]:
searcher = PubMedSearcher(email="albamaria.molero.perez@alumnos.upm.es")  # Replace with your email

PubMed searcher initialized!


In [ ]:
def create_causal_queries(var1: str, var2: str) -> List[str]:
    """Generate multiple query variations for causal relationships"""
    
    queries = [
        # Direct causal terms
        f"{var1} AND {var2} AND (causal OR causation OR cause)",
        f"{var1} AND {var2} AND (association OR relationship)",
        f"{var1} AND {var2} AND (risk factor OR predictor)",
        f"{var1} AND {var2} AND (correlation OR related)",
        
        # Methodological terms
        f"{var1} AND {var2} AND (longitudinal OR prospective)",
        f"{var1} AND {var2} AND (cohort study OR follow up)",
        f"{var1} AND {var2} AND (regression OR multivariate)",
        
        # Epidemiological terms  
        f"{var1} AND {var2} AND (epidemiology OR incidence)",
        f"{var1} AND {var2} AND (odds ratio OR hazard ratio)",
        
        # Simple broad search
        f"{var1} {var2}",
        f'"{var1}" AND "{var2}"'
    ]
    
    return queries

# Test causal relationship queries for age and diabetes
var1, var2 = "age", "diabetes mellitus"
causal_queries = create_causal_queries(var1, var2)

print(f"🔍 Testing causal relationship queries for: {var1} ↔ {var2}")
print("="*60)

best_query = None
max_results = 0

for i, query in enumerate(causal_queries, 1):
    print(f"\n{i:2d}. Query: {query}")
    pmids = searcher.search_papers(query, max_results=5)
    result_count = len(pmids)
    print(f"    Results: {result_count} papers")
    
    if result_count > max_results:
        max_results = result_count 
        best_query = query

print(f"\n🏆 Best query found: '{best_query}' with {max_results} results")

🔍 Testing causal relationship queries for: age ↔ diabetes mellitus

 1. Query: age AND diabetes mellitus AND (causal OR causation OR cause)
Found 5 papers for query: 'age AND diabetes mellitus AND (causal OR causation OR cause)'
    Results: 5 papers

 2. Query: age AND diabetes mellitus AND (association OR relationship)
Found 5 papers for query: 'age AND diabetes mellitus AND (association OR relationship)'
    Results: 5 papers

 3. Query: age AND diabetes mellitus AND (risk factor OR predictor)
Found 5 papers for query: 'age AND diabetes mellitus AND (risk factor OR predictor)'
    Results: 5 papers

 4. Query: age AND diabetes mellitus AND (correlation OR related)
Found 5 papers for query: 'age AND diabetes mellitus AND (correlation OR related)'
    Results: 5 papers

 5. Query: age AND diabetes mellitus AND (longitudinal OR prospective)
Found 5 papers for query: 'age AND diabetes mellitus AND (longitudinal OR prospective)'
    Results: 5 papers

 6. Query: age AND diabetes mellitus

In [ ]:
# GET ABSTRACTS FOR BEST QUERY

if best_query and max_results > 0:
    print(f"🔬 Fetching abstracts for best query: '{best_query}'")
    causal_papers = searcher.search_and_fetch(best_query, max_results=10)
    
    if causal_papers:
        print(f"\n📚 Retrieved {len(causal_papers)} papers on {var1} and {var2}:")
        print("="*70)
        
        for i, paper in enumerate(causal_papers[:3], 1):  # Show first 3
            print(f"\n{i}. {paper['title']}")
            print(f"   Authors: {paper['authors'][:100]}...")
            print(f"   Journal: {paper['journal']} ({paper['pub_date']})")
            print(f"   PMID: {paper['pmid']}")
            print(f"   Abstract: {paper['abstract'][:300]}...")
            print("-" * 50)
        
        # Save results for further analysis
        save_results_to_json(causal_papers, f"{var1}_{var2}_papers.json")
        
        # Create DataFrame
        df_causal = pd.DataFrame(causal_papers)
        print(f"\n✅ Created DataFrame with {len(df_causal)} papers")
        print("Columns:", df_causal.columns.tolist())
        
else:
    print("❌ No suitable queries found results. Try these alternatives:")
    
    alternative_terms = [
        ["aging", "diabetes"],
        ["elderly", "type 2 diabetes"], 
        ["older adults", "diabetic"],
        ["senescence", "hyperglycemia"],
        ["age related", "glucose metabolism"]
    ]
    
    print("\n🔄 Alternative term combinations to try:")
    for i, terms in enumerate(alternative_terms, 1):
        alt_query = f"{terms[0]} AND {terms[1]}"
        print(f"{i}. {alt_query}")
        
    print("\n💡 Tips:")
    print("- Use broader terms first, then narrow down")
    print("- Try synonyms (aging vs age, diabetes vs diabetic)")
    print("- Remove connecting words (and, or, with)")
    print("- Use MeSH terms for medical concepts")

🔬 Fetching abstracts for best query: 'age AND diabetes mellitus AND (causal OR causation OR cause)'
Searching PubMed for: 'age AND diabetes mellitus AND (causal OR causation OR cause)'
Found 10 papers for query: 'age AND diabetes mellitus AND (causal OR causation OR cause)'
Successfully retrieved 10 abstracts

📚 Retrieved 10 papers on age and diabetes mellitus:

1. Risk factors for type 2 diabetes mellitus.
   Authors: Barbara Fletcher; Meg Gulanick; Cindy Lamendola...
   Journal: The Journal of cardiovascular nursing (2002-Jan)
   PMID: 11800065
   Abstract: Genetic, environmental, and metabolic risk factors are interrelated and contribute to the development of type 2 diabetes mellitus. A strong family history of diabetes mellitus, age, obesity, and physical inactivity identify those individuals at highest risk. Minority populations are also at higher r...
--------------------------------------------------

2. Cardiovascular risk in diabetes mellitus: epidemiology, assessment and prev